# Audit unifié A1-A7 — Monde (même pipeline, France incluse)

Même fonction que le notebook *France* (`gbfs_toolkit.audit_static`, mêmes paramètres), appliquée au catalogue MobilityData **re-fetché en direct** via la librairie. Les colonnes FR et World deviennent donc comparables : un seul jeu de règles, une seule date.

> **Mise en garde.** Le re-fetch est live : c'est un *nouveau* snapshot (date du jour). Des flux du gel 2026-04 du papier sont morts en 2026-06 et tombent ; la couverture est rapportée explicitement. Pour le sweep complet, mettre `SAMPLE = None` (long, ~900 flux vivants).

In [1]:
import collections
import pandas as pd
import unified_audit as ua
CATALOG = '../experiments/e2_threshold_sensitivity/mobilitydata_systems.csv'
cat = ua.load_catalog(CATALOG)
print('Catalogue MobilityData :', len(cat), 'systèmes |',
      cat['country_code'].nunique(), 'pays | FR :', int((cat.country_code=='FR').sum()))

Catalogue MobilityData : 1509 systèmes | 48 pays | FR : 255


## 1. Échantillon (déterministe, FR + non-FR)

In [2]:
SAMPLE = 60   # None => sweep complet (long)
fr_ids  = cat.loc[cat.country_code=='FR', 'system_id'].dropna().astype(str).tolist()
nonfr   = cat.loc[cat.country_code!='FR', 'system_id'].dropna().astype(str).tolist()
if SAMPLE:
    ids = fr_ids[:SAMPLE//2] + nonfr[:SAMPLE - SAMPLE//2]
else:
    ids = cat['system_id'].dropna().astype(str).tolist()
print('À auditer :', len(ids), '(', sum(i in set(fr_ids) for i in ids), 'FR )')

À auditer : 60 ( 30 FR )


## 2. Fetch + audit (la fonction unique, identique à la France)

In [3]:
verdict, status, frame = ua.audit_world(ids, cat, max_workers=8)
ok = sum(1 for v in status.values() if v.startswith('ok'))
print(f'Couverture : {ok}/{len(ids)} flux audités ({100*ok/max(1,len(ids)):.0f}%)')
print('Stations auditées :', len(frame))

fetch_multiple: dott-klopein failed (failed to fetch https://gbfs.api.ridedott.com/public/v2/klopein/gbfs.json: 403 Client Error: Forbidden for url: https://gbfs.api.ridedott.com/public/v2/klopein/gbfs.json)


fetch_multiple: dott-velden-am-worthersee failed (failed to fetch https://gbfs.api.ridedott.com/public/v2/velden-am-worthersee/gbfs.json: 403 Client Error: Forbidden for url: https://gbfs.api.ridedott.com/public/v2/velden-am-worthersee/gbfs.json)


fetch_multiple: 511 failed (failed to fetch https://yaldi.rideatom.com/gbfs/511/v3.0/gbfs: HTTPSConnectionPool(host='yaldi.rideatom.com', port=443): Max retries exceeded with url: /gbfs/511/v3.0/gbfs (Caused by NameResolutionError("HTTPSConnection(host='yaldi.rideatom.com', port=443): Failed to resolve 'yaldi.rideatom.com' ([Errno -2] Name or service not known)")))


Couverture : 48/60 flux audités (80%)
Stations auditées : 8212


### Couverture détaillée (pourquoi des flux tombent)

In [4]:
br = collections.Counter(v.split(':')[0] for v in status.values())
pd.Series(dict(br)).sort_values(ascending=False)

ok                           48
empty station_information     9
unreachable                   3
dtype: int64

## 3. Comptes comparables FR vs World (même pipeline, échantillon)

In [5]:
sysf = ua.system_flags(verdict)
cmap = dict(zip(cat.system_id.astype(str), cat.country_code))
sysf['country'] = [cmap.get(i) for i in sysf.index]
world = ua.counts(sysf)
fr_s  = ua.counts(sysf[sysf.country=='FR'])
pd.DataFrame({'FR (echantillon)': fr_s, 'World (echantillon)': world})

,FR (echantillon),World (echantillon)
A1,0,0
A2,0,0
A3,14,37
A4,10,25
A5,0,1
A6,0,0
A7,11,26


## 4. Pour mémoire : France exacte (hors-ligne, sweep complet)

La même fonction, sur le catalogue FR publié, donne la colonne FR complète et exacte. Le World comparable complet s'obtient avec `SAMPLE = None` ci-dessus.

In [6]:
fr_full = ua.counts(ua.system_flags(ua.audit_france('../catalogue/stations_gold_standard_final.parquet')))
fr_full

{'A1': 17, 'A2': 1, 'A3': 41, 'A4': 88, 'A5': 4, 'A6': 0, 'A7': 32}

**Conclusion.** Une seule fonction d'audit produit les comptes FR et World. Pour intégrer au papier des chiffres World comparables, lancer le sweep complet (`SAMPLE = None`) : il remplace le snapshot 2026-04 par un audit unifié daté, où FR ⊂ World et toute classe se lit sous la même définition.